# RVU ID card detector: train the wearing / not wearing model

Runs on a free Colab T4 GPU in roughly 20 to 30 minutes. Use **Runtime > Run all**.

Data: Roboflow Universe `id-card-detection-ozsif/id-detectionn-2` version 1 (CC BY 4.0), 2,720 images, classes `id_not_wearing` and `id_wearing`.

Output: `rvu_idcard_best.onnx`, to be placed in the project's `models/` folder.

In [ ]:
!nvidia-smi -L || echo 'NO GPU: set Runtime > Change runtime type > T4 GPU'
!pip -q install ultralytics onnx onnxslim onnxruntime
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# Where the dataset zip comes from. Leave DATA_URL empty to be asked for a Roboflow API key.
DATA_URL = ""
import json, os, urllib.request
if not DATA_URL:
    from getpass import getpass
    key = getpass('Roboflow API key: ')
    api = 'https://api.roboflow.com/id-card-detection-ozsif/id-detectionn-2/1/yolov8?api_key=' + key
    DATA_URL = json.load(urllib.request.urlopen(api))['export']['link']
os.makedirs('/content/data', exist_ok=True)
urllib.request.urlretrieve(DATA_URL, '/content/data/ds.zip')
!cd /content/data && unzip -q -o ds.zip && rm ds.zip
open('/content/data/data.yaml', 'w').write(
    'path: /content/data\ntrain: train/images\nval: valid/images\ntest: test/images\n'
    'names:\n  0: id_not_wearing\n  1: id_wearing\n')
for split in ('train', 'valid', 'test'):
    print(split, len(os.listdir(f'/content/data/{split}/images')), 'images')

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='/content/data/data.yaml', epochs=60, patience=15, imgsz=640, batch=32,
            project='/content/runs', name='idcard', exist_ok=True, plots=True)

In [ ]:
best = YOLO('/content/runs/idcard/weights/best.pt')
metrics = best.val(data='/content/data/data.yaml', split='test')
print('RESULT test mAP50 = %.3f, mAP50-95 = %.3f' % (metrics.box.map50, metrics.box.map))
for i, name in best.names.items():
    print('RESULT %s: precision %.3f recall %.3f' % (name, metrics.box.p[i], metrics.box.r[i]))

In [ ]:
import shutil
onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True)
shutil.copy(onnx_path, '/content/rvu_idcard_best.onnx')
print('EXPORTED', os.path.getsize('/content/rvu_idcard_best.onnx') // 1024, 'KB')
from google.colab import files
files.download('/content/rvu_idcard_best.onnx')